# Tissue score comparison (limb vs neuron)

- Load limb and neuron VCF files produced by OpenSpliceAI.
- Parse the OpenSpliceAI INFO field into DS_* scores (max per allele/transcript).
- Align by variant (CHROM, POS, REF, ALT) and compute neuron - limb deltas.
- Summarize, visualize, and export the merged table.


In [ ]:
from pathlib import Path
from collections import defaultdict
import pandas as pd
import pysam

limb_vcf = Path('/home1/xyf/data/openspliceai_tissue_data/variant/limb_400.vcf')
neuron_vcf = Path('/home1/xyf/data/openspliceai_tissue_data/variant/neuron_400.vcf')

pd.set_option('display.max_columns', None)


In [ ]:
score_suffixes = ['AG', 'AL', 'DG', 'DL']


def parse_open_spliceai_entry(raw: str):
    parts = raw.split('|')
    if len(parts) < 10:
        return None
    allele, gene = parts[0], parts[1]
    values = {}
    for key, val in zip(
        ['DS_AG', 'DS_AL', 'DS_DG', 'DS_DL', 'DP_AG', 'DP_AL', 'DP_DG', 'DP_DL'],
        parts[2:10],
    ):
        try:
            values[key] = float(val) if key.startswith('DS') else int(val)
        except ValueError:
            values[key] = pd.NA
    return {'ALLELE': allele, 'GENE': gene, **values}


def aggregate_entries(entries):
    agg = {}
    for suf in score_suffixes:
        ds_key = f'DS_{suf}'
        vals = [e[ds_key] for e in entries if pd.notna(e[ds_key])]
        agg[ds_key] = max(vals) if vals else pd.NA
    valid_ds = [agg[f'DS_{s}'] for s in score_suffixes if pd.notna(agg[f'DS_{s}'])]
    agg['DS_MAX'] = max(valid_ds) if valid_ds else pd.NA
    return agg


def load_vcf_scores(path: Path) -> pd.DataFrame:
    rows = []
    vf = pysam.VariantFile(path)
    for rec in vf.fetch():
        if 'OpenSpliceAI' not in rec.info:
            continue
        parsed = [parse_open_spliceai_entry(raw) for raw in rec.info['OpenSpliceAI']]
        parsed = [p for p in parsed if p]
        if not parsed:
            continue
        alts = {str(a) for a in (rec.alts or [])}
        by_allele = defaultdict(list)
        for item in parsed:
            by_allele[item['ALLELE']].append(item)
        for allele, chunk in by_allele.items():
            if alts and allele not in alts:
                continue
            row = {
                'CHROM': str(rec.chrom),
                'POS': int(rec.pos),
                'REF': str(rec.ref),
                'ALT': allele,
            }
            row.update(aggregate_entries(chunk))
            rows.append(row)
    df = pd.DataFrame(rows)
    if not df.empty:
        df = (
            df.sort_values('DS_MAX', ascending=False)
            .drop_duplicates(subset=['CHROM', 'POS', 'REF', 'ALT'], keep='first')
            .reset_index(drop=True)
        )
    return df


In [ ]:
limb_df = load_vcf_scores(limb_vcf)
neuron_df = load_vcf_scores(neuron_vcf)

print(f"limb variants with scores: {len(limb_df):,}")
print(f"neuron variants with scores: {len(neuron_df):,}")
limb_df.head()


In [ ]:
merge_cols = ['CHROM', 'POS', 'REF', 'ALT']
merged = limb_df.merge(neuron_df, on=merge_cols, suffixes=('_limb', '_neuron'))

print(f"overlapping variants: {len(merged):,}")
merged.head()


In [ ]:
score_cols = ['DS_AG', 'DS_AL', 'DS_DG', 'DS_DL', 'DS_MAX']
for col in score_cols:
    merged[f'diff_{col}'] = merged[f'{col}_neuron'] - merged[f'{col}_limb']
    merged[f'abs_diff_{col}'] = merged[f'diff_{col}'].abs()

merged[merge_cols + [c for c in merged.columns if c.startswith('diff_DS')]].head()


In [ ]:
def summarize_diff(df: pd.DataFrame) -> pd.DataFrame:
    stats = []
    for col in score_cols:
        diff = df[f'diff_{col}'].dropna()
        stats.append({
            'score': col,
            'count': len(diff),
            'mean': diff.mean(),
            'median': diff.median(),
            'std': diff.std(),
            'min': diff.min(),
            'max': diff.max(),
            'abs_p90': diff.abs().quantile(0.9),
            'abs_p99': diff.abs().quantile(0.99),
        })
    return pd.DataFrame(stats).set_index('score')

diff_stats = summarize_diff(merged)
diff_stats


In [ ]:
top = merged.sort_values('abs_diff_DS_MAX', ascending=False)
cols_to_show = merge_cols + [
    'DS_MAX_limb', 'DS_MAX_neuron', 'diff_DS_MAX',
    'DS_AG_limb', 'DS_AG_neuron', 'diff_DS_AG',
    'DS_AL_limb', 'DS_AL_neuron', 'diff_DS_AL',
    'DS_DG_limb', 'DS_DG_neuron', 'diff_DS_DG',
    'DS_DL_limb', 'DS_DL_neuron', 'diff_DS_DL',
]
top[cols_to_show].head(20)


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(score_cols), figsize=(4 * len(score_cols), 3), constrained_layout=True)
for ax, col in zip(axes, score_cols):
    diff = merged[f'diff_{col}'].dropna()
    ax.hist(diff, bins=50, color='steelblue', edgecolor='white')
    ax.axvline(0, color='red', linestyle='--', linewidth=1)
    ax.set_title(col)
    ax.set_xlabel('neuron - limb')
    ax.set_ylabel('count')
plt.show()


In [ ]:
out_dir = Path('T')
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / 'limb_neuron_score_diff.parquet'
merged.to_parquet(out_path, index=False)
print(f'saved merged diff table to {out_path}')
